# Model Evaluation

This notebook covers:
- Model testing on test set
- Confusion matrix and metrics
- Per-class performance analysis
- Misclassification analysis

## Import Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torchvision import datasets, transforms, models
from torchvision.models import MobileNet_V3_Large_Weights
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import random

import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Load Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Find the latest saved model
models_dir = Path('../models')
saved_models = sorted(models_dir.glob('plant_disease_mobilenetv3_best_*.pth'))

if not saved_models:
    raise FileNotFoundError("No saved model found. Run 03_model_training.ipynb first.")

model_path = saved_models[-1]  # Use the latest
print(f"Loading model: {model_path.name}")

# Load checkpoint
checkpoint = torch.load(model_path, map_location=device)
classes = checkpoint['classes']
num_classes = checkpoint['num_classes']

# Rebuild model architecture
model = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Linear(960, 1280),
    nn.Hardswish(),
    nn.Dropout(p=0.3),
    nn.Linear(1280, num_classes)
)

model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"Model loaded successfully")
print(f"Classes ({num_classes}): {classes}")

## Run Predictions on Validation Set

In [ ]:
# Validation data loader
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

val_dataset = datasets.ImageFolder('../data/val', transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2, pin_memory=True)

print(f"Validation samples: {len(val_dataset):,}")

# Collect all predictions
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

overall_acc = 100 * np.sum(all_preds == all_labels) / len(all_labels)
print(f"Overall Accuracy: {overall_acc:.2f}%")

## Classification Report

In [ ]:
# Precision, Recall, F1 per class
print("Classification Report")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=classes))

## Confusion Matrix

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Per-Class Accuracy

In [ ]:
# Per-class accuracy from confusion matrix diagonal
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

print("Per-Class Accuracy:")
print("-" * 40)
for cls, acc in zip(classes, per_class_acc):
    bar = '#' * int(acc / 5)
    print(f"  {cls:<40} {acc:6.2f}%  {bar}")

# Bar chart
plt.figure(figsize=(12, 6))
colors = ['green' if a >= 90 else 'orange' if a >= 75 else 'red' for a in per_class_acc]
plt.barh(classes, per_class_acc, color=colors, alpha=0.8)
plt.xlabel('Accuracy (%)')
plt.title('Per-Class Accuracy', fontsize=14, fontweight='bold')
plt.axvline(x=90, color='green', linestyle='--', alpha=0.5, label='90% threshold')
plt.xlim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

## Sample Predictions

In [ ]:
def predict_image(image_path):
    """Run inference on a single image and return predicted class and confidence"""
    img = Image.open(image_path).convert('RGB')
    tensor = val_transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probs, 1)
    
    return classes[predicted.item()], confidence.item() * 100, img


# Display sample correct and incorrect predictions
val_dir = Path('../data/val')
correct_samples = []
incorrect_samples = []

for class_name in classes:
    class_dir = val_dir / class_name
    images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
    
    for img_path in random.sample(images, min(10, len(images))):
        pred_class, confidence, img = predict_image(img_path)
        entry = (img, class_name, pred_class, confidence)
        
        if pred_class == class_name and len(correct_samples) < 6:
            correct_samples.append(entry)
        elif pred_class != class_name and len(incorrect_samples) < 6:
            incorrect_samples.append(entry)
        
        if len(correct_samples) >= 6 and len(incorrect_samples) >= 6:
            break

# Plot correct predictions
print("Correct Predictions (sample):")
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for idx, (img, actual, pred, conf) in enumerate(correct_samples[:6]):
    axes[idx//3][idx%3].imshow(img)
    axes[idx//3][idx%3].set_title(f"Actual: {actual}\nPred: {pred} ({conf:.1f}%)", 
                                   fontsize=8, color='green')
    axes[idx//3][idx%3].axis('off')
plt.suptitle('Correct Predictions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Plot incorrect predictions
if incorrect_samples:
    print("Incorrect Predictions (misclassified):")
    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    for idx, (img, actual, pred, conf) in enumerate(incorrect_samples[:6]):
        axes[idx//3][idx%3].imshow(img)
        axes[idx//3][idx%3].set_title(f"Actual: {actual}\nPred: {pred} ({conf:.1f}%)", 
                                       fontsize=8, color='red')
        axes[idx//3][idx%3].axis('off')
    plt.suptitle('Incorrect Predictions (Misclassified)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No incorrect predictions found in sample!")

## Summary

In [ ]:
print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"\nModel: {model_path.name}")
print(f"Overall Accuracy: {overall_acc:.2f}%")
print(f"\nPer-Class Breakdown:")

for cls, acc in zip(classes, per_class_acc):
    status = "Good" if acc >= 90 else "Needs improvement" if acc >= 75 else "Poor"
    print(f"  {cls:<40} {acc:6.2f}%  ({status})")

weak_classes = [cls for cls, acc in zip(classes, per_class_acc) if acc < 90]
if weak_classes:
    print(f"\nClasses below 90%: {weak_classes}")
    print("Consider: More training data or additional augmentation for these classes")
else:
    print("\nAll classes above 90% - model is performing well!")

print(f"\nNext step: Proceed to 05_model_conversion.ipynb")